# 02_data_cleaning.ipynb
Clean the synthetic datasets by checking missing values, removing duplicates, fixing data types, and handling outliers. This notebook runs independently.

In [7]:
from pathlib import Path
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from faker import Faker
warnings.filterwarnings('ignore')
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'
GENERATED_DATA_DIR = BASE_DIR / 'data' / 'generated'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import src.generate_data as generate_data
import src.cleaning as cleaning
print('Setup complete')

Setup complete


In [8]:
# Ensure generated data exists before cleaning
csv_files = sorted(GENERATED_DATA_DIR.glob('*.csv'))
if not csv_files:
    generate_data.generate_all(n_each=1000, out_dir=GENERATED_DATA_DIR)
    csv_files = sorted(GENERATED_DATA_DIR.glob('*.csv'))
[p.name for p in csv_files]

['barangay_census.csv',
 'medical_records.csv',
 'sales_orders.csv',
 'school_enrollment.csv']

In [9]:
# Load all datasets
datasets = {p.stem: pd.read_csv(p) for p in csv_files}
{name: df.shape for name, df in datasets.items()}

{'barangay_census': (1020, 7),
 'medical_records': (1020, 10),
 'sales_orders': (1020, 10),
 'school_enrollment': (1020, 10)}

In [10]:
# Check missing values
for name, df in datasets.items():
    print(f'\n=== Missing values: {name} ===')
    display(cleaning.report_missing(df).rename('missing_count').to_frame())


=== Missing values: barangay_census ===


,missing_count
households,50
population,50
median_income,50
record_id,0
barangay,0
city,0
collection_date,0



=== Missing values: medical_records ===


,missing_count
age,50
temperature_c,50
cholesterol_mgdl,50
weight_kg,50
visit_id,0
visit_date,0
patient_name,0
sex,0
barangay,0
diagnosis,0



=== Missing values: sales_orders ===


,missing_count
quantity,50
unit_price,50
total_amount,50
order_id,0
order_date,0
customer_name,0
barangay,0
product,0
category,0
payment_method,0



=== Missing values: school_enrollment ===


,missing_count
math_score,50
english_score,50
science_score,50
attendance_rate,50
student_id,0
student_name,0
birth_date,0
grade_level,0
school_name,0
barangay,0


In [11]:
# Clean sales orders as the main example
sales = datasets['sales_orders'].copy()
sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')
sales['quantity'] = pd.to_numeric(sales['quantity'], errors='coerce')
sales['unit_price'] = pd.to_numeric(sales['unit_price'], errors='coerce')
sales['total_amount'] = pd.to_numeric(sales['total_amount'], errors='coerce')
sales, removed_dupes = cleaning.drop_duplicates(sales)
sales = cleaning.fill_numeric_with_median(sales, ['quantity', 'unit_price', 'total_amount'])
for column in ['quantity', 'unit_price', 'total_amount']:
    sales = cleaning.cap_outliers_iqr(sales, column)
print('Removed duplicates:', removed_dupes)
display(sales.head())

Removed duplicates: 20


,order_id,order_date,customer_name,barangay,product,category,quantity,unit_price,total_amount,payment_method
0,SO-100383,2026-05-24,Dawn Hall,San Jose,Canned Sardines,Beverage,3.0,401.39,1204.17,Cash
1,SO-100775,2026-05-24,Carol Wiggins,San Miguel,Canned Sardines,Personal Care,4.0,174.91,699.64,GCash
2,SO-100791,2026-05-24,John Barron MD,Maligaya,Rice,Grocery,6.0,289.09,1734.54,Cash
3,SO-100131,2026-05-24,Michelle Spears,San Isidro,Canned Sardines,Household,5.0,77.65,388.25,Bank Transfer
4,SO-100464,2026-05-24,Anita Lindsey,Rizal,Cooking Oil,Household,1.0,395.20,395.20,Card


In [12]:
# Save cleaned datasets
for name, df in datasets.items():
    cleaned = df.copy()
    if name == 'sales_orders':
        cleaned = sales
    cleaned.to_csv(PROCESSED_DATA_DIR / f'{name}_cleaned.csv', index=False)
print('Saved cleaned files to data/processed')

Saved cleaned files to data/processed


## Cleaning notes
The notebook now produces cleaned CSV files in `data/processed/` and can be rerun independently.